# Structured output

Models can be requested to provide their response in a format matching a given schema. This is useful for ensuring the output can be easily parsed and used in subsequent processing. LangChain supports multiple schema types and methods for enforcing structured output.

# Pydantic

Pydantic models provide the richest feature set with field validation, descriptions, and nested structures.


In [2]:
import os
from langchain.chat_models import init_chat_model

os.environ["GROQ_API_KEY"]= os.getenv("GROQ_API_KEY")
model = init_chat_model("groq:qwen/qwen3.8-27b")
model


ChatGroq(metadata={'lc_versions': {'langchain-core': '1.6.3', 'langchain': '1.4.0'}}, client=<groq.resources.chat.completions.Completions object at 0x783b235ab050>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x783b232bf4a0>, model_name='qwen/qwen3.8-27b', model_kwargs={}, groq_api_key=SecretStr('**********'))

In [ ]:
from pydantic import BaseModel,Field

class Movie(BaseModel):
    title:str=Field(description="The title of the movie")
    year:int=Field(description="This year the movie was released")
    director:str=Field(description="The director of the movie")
    rating:float=Field(description="The movies rating out of 10")

In [4]:
model_with_structure = model.with_structured_output(Movie)
model_with_structure

_ChatModelBinding(bound=ChatGroq(metadata={'lc_versions': {'langchain-core': '1.6.3', 'langchain': '1.4.0'}}, client=<groq.resources.chat.completions.Completions object at 0x77f3b2e192b0>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x77f3b2ef30e0>, model_name='qwen/qwen3.8-27b', model_kwargs={}, groq_api_key=SecretStr('**********')), kwargs={'tools': [{'type': 'function', 'function': {'name': 'Movie', 'description': '', 'parameters': {'properties': {'title': {'description': 'The title of the movie', 'type': 'string'}, 'year': {'description': 'This year the movie was released', 'type': 'integer'}, 'director': {'description': 'The director of the movie', 'type': 'string'}, 'rating': {'description': 'The movies rating out of 10', 'type': 'number'}}, 'required': ['title', 'year', 'director', 'rating'], 'type': 'object'}}}], 'ls_structured_output_format': {'kwargs': {'method': 'function_calling'}, 'schema': {'type': 'function', 'function': {'name': 'Movie', 'de

In [5]:
model.invoke("Provide Details about the movie Inception")

AIMessage(content='**Inception** is a 2010 science fiction action film directed by and written by **Christopher Nolan**. It is widely regarded as one of the most complex and influential films of the 21st century, known for its layered narrative, mind-bending concepts, and high-stakes heist structure.\n\n### Basic Information\n- **Release Date:** July 16, 2010\n- **Studio:** Legendary Pictures, Syncopy Inc., Warner Bros. Pictures\n- **Runtime:** 148 minutes\n- **Genre:** Sci-Fi, Action, Thriller\n- **Music:** Hans Zimmer (featuring "Memory" and "Time")\n- **Box Office:** ~$836 million worldwide\n\n### Plot Summary\nThe story follows **Dom Cobb** (Leonardo DiCaprio), a skilled thief who specializes in "inception" — the act of planting an idea in a person’s subconscious while they are dreaming. This is considered the hardest and most dangerous form of theft in the dream world.\n\nCobb has been living in exile because he is falsely accused of killing his wife, **Mal** (Marion Cotillard), w

In [8]:
response = model_with_structure.invoke("Provide Details about the movie Inception")
response

Movie(title='Inception', year=2010, director='Christopher Nolan', rating=8.8)

### Message output slongside parsed structure

In [9]:
from pydantic import BaseModel,Field

class Movie(BaseModel):
    """A movie with details."""
    title:str=Field(...,description="The title of the movie")
    year:int=Field(...,description="This year the movie was released")
    director:str=Field(...,description="The director of the movie")
    rating:float=Field(...,description="The movies rating out of 10")

model_with_structure = model.with_structured_output(Movie, include_raw=True)

response = model_with_structure.invoke("Provide Details about the movie Inception")
response

{'raw': AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'dzje7bmkb', 'function': {'arguments': '{"director":"Christopher Nolan","rating":8.8,"title":"Inception","year":2010}', 'name': 'Movie'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 68, 'prompt_tokens': 354, 'total_tokens': 422, 'completion_time': 0.176715365, 'completion_tokens_details': None, 'prompt_time': 0.02419409, 'prompt_tokens_details': None, 'queue_time': 0.05356185, 'total_time': 0.200909455}, 'model_name': 'qwen/qwen3.8-27b', 'system_fingerprint': 'fp_21e59ac2de', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--01a0aa48-6cf2-76c2-bd89-afdff56fafab-0', tool_calls=[{'name': 'Movie', 'args': {'director': 'Christopher Nolan', 'rating': 8.8, 'title': 'Inception', 'year': 2010}, 'id': 'dzje7bmkb', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 354, 'output_tokens': 68, 'total_t

### Nested Structure



In [3]:
from pydantic import BaseModel, Field

class Actor(BaseModel):
    name:str
    role:str

class MovieDetails(BaseModel):
    title: str
    year: int
    cast: list[Actor]
    genres: list[str]
    budget: float | None = Field(None, description="Budget in millions USD")

model_with_structure = model.with_structured_output(MovieDetails)

response = model_with_structure.invoke("Provide details about he movie Inception")

response





MovieDetails(title='Inception', year=2010, cast=[Actor(name='Leonardo DiCaprio', role='Cobb'), Actor(name='Joseph Gordon-Levitt', role='Arthur'), Actor(name='Elliot Page', role='Ariadne'), Actor(name='Tom Hardy', role='Eames'), Actor(name='Ken Watanabe', role='Saito'), Actor(name='Marion Cotillard', role='Mal'), Actor(name='Michael Caine', role='Miles'), Actor(name='Tom Berenger', role='Browning'), Actor(name='Cillian Murphy', role='Robert Fischer'), Actor(name='Dileep Rao', role='Yusuf')], genres=['Action', 'Science Fiction', 'Thriller'], budget=160.0)

### TypedDict

TypedDict provides a simpler alternative using Python's built-in typing , ideal when you don't need runtime validation.